In [ ]:
import numpy as np
%matplotlib TkAgg
import matplotlib.pyplot as plt

import radar
import cfar

In [ ]:
def cargar_matriz(fname,dtype=np.complex128):
    dim = np.loadtxt(fname,max_rows=1,dtype=np.int_)
    return np.reshape(np.loadtxt(fname,skiprows=1,dtype=dtype),shape=dim)

In [ ]:
# chip, 1
pulso=cargar_matriz("signal_tx.txt")

# chip
pulso = np.reshape(pulso,(-1,))

# Antena, muestra, pulso
datos=cargar_matriz("Matrix_rx.txt")
# pulso, antena, muestra
datos = np.transpose(datos,(2,0,1))

In [ ]:
import importlib
importlib.reload(radar)

In [ ]:
radar_ = radar.Canal_radar(60e3,10e6,pulso,0,350/60e3*3e8,0)

In [ ]:
prf = 150 # Frecuencia de repetición de pulso

In [ ]:
compresor = radar.Compresor_pulso(radar_)

In [ ]:
def fig16(exploracion,comp_pulso,fig=16):
    # pulso, antena, muestra
    plt.figure(fig,figsize=(10,10))
    doppler_rango = np.fft.fft(exploracion,axis=0)
    corr =  np.array([comp_pulso.amplitud_cuadrada_correlacion(doppler_rango[i,0,:]) for i in range(exploracion.shape[0])])
    corr = np.fft.fftshift(corr,0)
    R,F = np.meshgrid(np.arange(corr.shape[1]),np.fft.fftshift(np.fft.fftfreq(corr.shape[0],0.5)))
    #ax=plt.axes(projection='3d')
    #ax.plot_surface(R,F,corr)
    plt.contourf(R,F,10*np.log10(corr))
    plt.show()
fig16(datos,compresor,fig=1)

In [ ]:
def comp_e_integra_pulsos(exploracion,comp_pulso):
    """ Calcula la integración coherente y correlación en tiempo corto
        
        exploración: Muestras de cada pulso y antena dimensión (pulsos, antenas, muestras)
        comp_pulso: compresor de pulso para el código del radar
    """
    muestras_doppler_antena_rango = np.fft.fftshift(np.fft.fft(exploracion,axis=0),0)
    nfrecs,nantenas,nmuestras = muestras_doppler_antena_rango.shape

    acuad_doppler_antena_rango = np.zeros((nfrecs,nantenas,nmuestras+comp_pulso.pulso.size-1),dtype=float)
    for ifrec in range(nfrecs):
        for iant in range(nantenas):
            acuad_doppler_antena_rango[ifrec,iant,:] = comp_pulso.amplitud_cuadrada_correlacion(muestras_doppler_antena_rango[ifrec,iant,:])
    return acuad_doppler_antena_rango

correlacion = comp_e_integra_pulsos(datos, compresor)[:,0,:]

In [ ]:
def predictor_adaptativo(datos,respuesta_deseada,coeficientes,potencia,mu,alpha, estado):
    """ Filtro predictor adaptativo
    @param datos arreglo de muestras a procesar
    @param respuesta_deseada salida a predecir (una muestra en avance respecto a datos)
    @param coeficientes vector de coeficientes, actualizados en cada invocación
    @param potencia vector de dimensión 1, estimado de potencia de la señal, actualizado en cada invocación
    @param mu parámetro de convergencia, determina cuánto cambian en proporción los coeficientes por el error
    @param alpha factor de olvido, determina cuánto cambia el estimado de potencia en función de la potencia instantánea
    @param estado estado del filtro, del mismo tamaño que coeficientes
    @return arreglo de predicciones correspondientes a los datos (uno en avance)
    """
    # estado[1:] = estado[:-1]
    # estado[0] = datos[k]
    # error_actual = respuesta_deseada[k] - sum(estado*coeficientes)
    # potencia = alpha * datos[k] ** 2 + (1 - alpha) * potencia
    # coeficientes = coeficientes + 2 * mu * error_actual * estado / (estado.size * potencia)

    if (estado.shape != coeficientes.shape):
        raise ValueError("Forma de estado no coincide con forma de coeficientes")
    if (mu <= 0.0 or mu >= 1.0):
        raise ValueError("Parámetro de convergencia fuera de rango")
    if (alpha < 0.0 or alpha >= 1):
        raise ValueError("Factor de olvido fuera de rango")
    if (potencia <= 0.0):
        raise ValueError("Estimado de potencia fuera de rango")
    if (datos.shape != respuesta_deseada.shape):
        raise ValueError("Formas de respuesta_deseada y datos no coinciden")
    salida = np.zeros_like(datos)
    for k in range(datos.size):
        estado[1:] = estado[:-1]
        estado[0] = datos[k]
        salida[k] = np.sum(estado*coeficientes)
        error_actual = respuesta_deseada[k] - salida[k]
        potencia[:] = alpha * np.abs(datos[k]) ** 2 + (1 - alpha) * potencia
        coeficientes[:] = coeficientes + 2 * mu * error_actual * estado / (estado.size * potencia)
    return salida

In [ ]:
def prueba_predictor(mu=.1,alpha=.01,iteraciones=2,orden=100,fig=1):
    t = np.linspace(0,1,1000)
    x = np.sin(10*2*np.pi*t)+np.sin(3*2*np.pi*t)
    err = np.random.normal(loc=0,scale=1,size=1000)
    z = x + err
    coeficientes = np.zeros(orden)
    potencia = np.array([.1])
    estado = np.zeros(orden)
    for _ in range(iteraciones):
        estado[:]=0
        salida = predictor_adaptativo(z[:-1],z[1:],coeficientes,potencia,mu,alpha,estado)
    plt.figure(fig,layout="tight")
    plt.clf()
    plt.subplot(2,1,1)
    plt.plot(t,z)
    plt.plot(t,x,'--')
    plt.title("Entrada y señal sin ruido")
    plt.subplot(2,1,2)
    plt.plot(t[1:],salida)
    plt.plot(t[1:],x[1:],'--')
    plt.title("Salida y señal sin ruido")
    plt.suptitle("Prueba de Predictor Adaptativo")
    return coeficientes

_=prueba_predictor(mu=0.01,alpha=0.01,iteraciones=2,orden=300,fig=2)

In [ ]:
def suprime_clutter(datos,mu=.01,alpha=.01,orden=100):
    estado = np.zeros(orden,dtype=datos.dtype)
    coeficientes = np.zeros(orden,dtype=datos.dtype)
    potencia = np.array([.1],dtype = datos.dtype)
    salida = np.zeros_like(datos)
    datos = np.concatenate((np.zeros((1,*datos.shape[1:]),dtype=datos.dtype),datos),axis=0)
    for i in range(datos.shape[1]):
        for j in range(datos.shape[2]):
            estado[:] = 0
            salida[:,i,j] = predictor_adaptativo(datos[:-1,i,j],datos[1:,i,j],coeficientes,potencia,mu,alpha,estado)
    return datos[1:,:,:] - salida

datos_sup_clutter = suprime_clutter(datos,orden=300,alpha=.1,mu=.001)
correlacion_sup_clutter = comp_e_integra_pulsos(datos_sup_clutter,compresor)[:,0,:]

In [ ]:
plt.figure(3,layout="tight")
plt.clf()
plt.matshow(correlacion[:,compresor.pulso.size-1:-compresor.pulso.size+1],fignum=3)
plt.title("Doppler-rango")
plt.colorbar()

In [ ]:
def guardar_matriz_en_archivo(matriz,archivo):
    np.savetxt(archivo,matriz,delimiter=",")

guardar_matriz_en_archivo(correlacion[:,compresor.pulso.size-1:-compresor.pulso.size+1],"mag_cuadrada.csv")

In [ ]:
A=np.loadtxt("mag_cuadrada-2026_11_20.csv",delimiter=",")

In [ ]:
A.shape

In [ ]:
plt.figure(4,layout="tight")
plt.clf()
plt.matshow(correlacion_sup_clutter[:,compresor.pulso.size-1:-compresor.pulso.size+1],fignum=4)
plt.title("Doppler-rango, con supresión de clutter")
plt.colorbar()

In [ ]:
plt.figure(5)
plt.title(f"Doppler 176 (frecuencia {np.fft.fftshift(np.fft.fftfreq(correlacion.shape[0],1/prf))[176]:0.3} Hz)")
plt.plot(correlacion[176,compresor.pulso.size-1:-compresor.pulso.size+1])
plt.plot(correlacion_sup_clutter[176,compresor.pulso.size-1:-compresor.pulso.size+1])
plt.show()

In [ ]:
param_radar =  {
    "frec_muestreo": 1,
    "ancho_pulso": radar_.pulso.size,
    "distancia_ciega": 0,
    "rango_por_unidad_de_retardo": 1
} 
param_detector ={
    "ancho_ventana": radar_.pulso.size*2,
    "T": cfar.calcula_t_para_pfa(radar_.pulso.size*2)
}

In [ ]:
def detectar(correlacion,param_radar,param_detector):
    detecciones=[]
    for i in range(correlacion.shape[0]):
        rangos,_=cfar.detectarOsCfar(param_radar,correlacion[i,:],param_detector)
        if rangos:
            detecciones.append((i,rangos))
    return detecciones

detecciones = detectar(correlacion,param_radar,param_detector)
detecciones_sup_clutter = detectar(correlacion_sup_clutter,param_radar,param_detector)

In [ ]:
detecciones

In [ ]:
detecciones_sup_clutter

In [ ]:
plt.plot(A[176,:],'x--')

In [ ]:
def examina_doppler(correlacion,param_radar,param_detector,n,fig=1):
    dets,x = cfar.detectarOsCfar(param_radar,correlacion[n,:],param_detector)
    plt.figure(fig,layout="tight")
    plt.plot(correlacion[n,:],"x-")
    plt.plot(np.array(x["centros"]),np.array(x["umbrales"]),"o-")
    print(dets)
examina_doppler(correlacion,param_radar,param_detector,176,6)
plt.title("Señal y umbral cfar para doppler 176")
examina_doppler(correlacion_sup_clutter,param_radar,param_detector,176,7)
plt.title("Señal con supresión de clutter y umbral cfar para doppler 176")

In [ ]:
plt.figure(8,layout="tight")
plt.subplot(2,1,1)
plt.plot(compresor.pulso,"o--")
plt.title("Código")
plt.subplot(2,1,2)
plt.plot(compresor.correlacion(compresor.pulso),"o--")
plt.title("Autocorrelación del código")

In [ ]:
plt.figure(9,layout="tight")
(lambda x:plt.plot(np.correlate(x,x,mode="full"),"o--"))(np.array([1,1,1,-1,-1,-1,1,-1,-1,1,-1]))
plt.title("Autocorrelación de un código barker 11")

In [ ]:
plt.figure(9)
plt.clf()
plt.subplot(2,1,1)
plt.plot(np.real(datos[:,0,176]))
plt.plot(np.real(datos[:,0,176]-datos_sup_clutter[:,0,176]),"--")
plt.subplot(2,1,2)
plt.plot(np.imag(datos[:,0,176]))
plt.plot(np.imag(datos[:,0,176]-datos_sup_clutter[:,0,176]),"--")